In [ ]:
import xarray as xr
import metview as mv
import pystac_client
import itertools
import time
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import os
os.environ['PROJ_LIB'] = "/opt/conda/envs/geospatial/share/proj"

xr.set_options(keep_attrs=True)

In [ ]:
service = pystac_client.Client.open('https://data.inpe.br/bdc/stac/v1/')
service

In [ ]:
for collection in service.get_collections():
    print(collection)

In [ ]:
collection = service.get_collection('prec_merge_daily-1')
collection

In [ ]:
for item in itertools.islice(collection.get_items(), 10):
    print(f"{item.id}")

In [ ]:
item

In [ ]:
item.assets

In [ ]:
url = item.assets['merge_daily'].href
url

In [ ]:
item_search = service.search(datetime='2025-01-01/2025-01-31',
                             collections=['prec_merge_daily-1'])

# Create tmp folder or remove existing mars files
tmp_folder = "Dados/tmp/"
if not os.path.exists(tmp_folder):
    os.makedirs(tmp_folder)
else:
    for file in os.listdir(tmp_folder):
        path = os.path.join(tmp_folder, file)
        if os.path.isfile(path):
            os.remove(path)
        elif os.path.isdir(path):
            os.rmdir(path)

ds = mv.Fieldset()
for i in item_search.items():
    url = i.assets['merge_daily'].href
    time.sleep(0.1)
    print(url)
    ds.append(mv.download (url=url, target=tmp_folder))
    
print (i.assets['ctl'].href)

In [ ]:
ds.describe()

In [ ]:
area = [-60,-120,20,0] # S,W,N,E
AMS = mv.geoview(
    map_area_definition = "corners",
    area                = area,
    coastlines = mv.mcoast(
        map_coastline_land_shade        = "on",
        map_coastline_land_shade_colour = "#eeeeee",
        map_grid_latitude_increment     = 10,
        map_grid_longitude_increment    = 10)
    )

auto_style = mv.mcont(contour_automatic_setting = "ecmwf")


In [ ]:
shaded = mv.mcont(
    legend                       = "on",
    contour_highlight            = "off",
    contour_level_selection_type = "level_list",
    contour_level_list           = [2.5, 5, 10, 15, 35, 50, 75, 100, 125, 150, 200, 300],
    contour_label                = "off",
    contour_shade                = "on",
    contour_shade_method         = "area_fill",
    contour_line_colour_rainbow_direction = "clockwise",
    contour_line_colour_rainbow  = "on"
#    contour_shade_colour_list    = ["sky","greenish_blue","avocado",
#                                    "orange","orangish_red","violet", "RGB(1,0,0)"]
    )

In [ ]:
#ds = ds.to_dataset()
#ds
mv.plot(AMS, ds['prec'], shaded)

In [ ]:
#xds = mv.to_dataset(ds)


In [ ]:
## lats, lons = mv.latitudes(ds['prec']).reshape(31, 1001, 924), mv.longitudes(ds['prec']).reshape(31, 1001, 924)
merge_array = mv.values(ds['prec']).reshape(31, 924, 1001)
lats, lons = mv.latitudes(ds['prec']).reshape(31, 924, 1001), mv.longitudes(ds['prec']).reshape(31, 924, 1001)
times=mv.valid_date(ds['prec'])

print (lats.shape, lons.shape, merge_array.shape, np.size(times))


In [ ]:
#xds = ds['prec'].to_dataset()


In [ ]:
# Definir níveis de contorno personalizados
levels = [2.5, 5, 10, 12.5, 15, 20, 25, 30, 50]

# Criar a figura e os eixos com projeção de mapa
fig, ax = plt.subplots(
    figsize=(10, 6),
    subplot_kw={"projection": ccrs.PlateCarree()}  # Projeção geográfica
)

# Plotar os dados de precipitação (preenchimento)
contourf = ax.contourf(lons[0], lats[0], merge_array.mean(axis=0), 
#                      levels=levels, 
                      cmap='BuPu', 
                      transform=ccrs.PlateCarree())

# Adicionar rótulos aos contornos
ax.clabel(contourf, #levels=levels, 
         inline=True, 
         fontsize=8, 
         fmt='%1.0f')

# Adicionar barra de cores
cbar = plt.colorbar(contourf, ax=ax, orientation='horizontal', pad=0.05)
cbar.set_label('Precipitação (mm)')

# Adicionar elementos do mapa
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.gridlines(draw_labels=True, linestyle='--')

# Definir título
ax.set_title('Mapa de Precipitação com Contornos')

plt.show()

In [ ]:
xds = xr.Dataset(
    data_vars={
        "prec": (("time", "lat", "lon"), merge_array),
    },
    coords={
        "time": times[:31],
        "lat": (("y", "x"), lats[0]),
        "lon": (("y", "x"), lons[0]),
    }
)

In [ ]:
xds

In [ ]:
xds.prec.head()


In [ ]:
xds.prec.plot()

In [ ]:
xds.prec[0].shape


In [ ]:
# Supondo que você já tenha um dataset xarray chamado `ds`
# Exemplo de extração de dados
p = xds['prec']  # Variável de precipitação
y = xds['lat']    # Coordenadas de latitude
x = xds['lon']    # Coordenadas de longitude

# Criar a figura e os eixos com projeção de mapa
fig, ax = plt.subplots(
    figsize=(10, 6),
    subplot_kw={"projection": ccrs.PlateCarree()}  # Projeção geográfica
)

# Plotar os dados de precipitação (preenchimento)
contourf = ax.contourf(x, y, p.mean(axis=0), 
                      cmap='BuPu', 
                      transform=ccrs.PlateCarree())

# Adicionar rótulos aos contornos
ax.clabel(contourf, #levels=levels, 
         inline=True, 
         fontsize=8, 
         fmt='%1.0f')

# Adicionar barra de cores
cbar = plt.colorbar(contourf, ax=ax, orientation='horizontal', pad=0.05)
cbar.set_label('Precipitação (mm)')

# Adicionar elementos do mapa
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.gridlines(draw_labels=True, linestyle='--')

# Definir título
ax.set_title('Mapa de Precipitação com Contornos')

plt.show()